# Proactive Wait-Time Model

這個 notebook 會直接完成以下流程：

- 訓練一個不依賴 LLM 的輕量等待時長模型
- 跑完整 benchmark 與黃金樣例驗收
- 產出 `artifacts/metrics.json`、`artifacts/benchmark_report.md`、`artifacts/wait_policy_bundle.pt`
- 在 Colab 中把權重下載到本地

In [1]:
from run_training import train_and_benchmark

metrics = train_and_benchmark(seed=7)
metrics

{'run': {'seed': 7,
  'device': 'cuda',
  'elapsed_seconds': 200.83,
  'selected_trial': {'name': 'large',
   'hidden_dims': (384, 192, 96),
   'dropout': 0.1,
   'lr': 0.0009,
   'batch_size': 384,
   'epochs': 10,
   'weight_decay': 0.0001},
  'best_epoch': 8,
  'threshold': 0.42},
 'model': {'size': 24000,
  'exact_bucket_acc': 0.95675,
  'within_one_bucket_acc': 0.9886666666666667,
  'followup_precision': 1.0,
  'followup_recall': 0.99697950040644,
  'followup_f1': 0.9984850856896599,
  'mae_seconds': 3449.6777888432302,
  'median_abs_error_seconds': 0.0,
  'p90_abs_error_seconds': 483.66666666666697,
  'mape': 0.15238612714937044,
  'adaptive_15s_rate': 0.9248752214160889,
  'adaptive_60s_rate': 0.9659351842887597,
  'adaptive_300s_rate': 0.9816170225976718,
  'short_mae_seconds': 1.7646074854871319,
  'short_within_5s_rate': 0.9768996421645081,
  'false_early_rate': 0.012824950651294884,
  'false_late_rate': 0.008753610855432636,
  'practical_score': 0.9702822762507401},
 'model_

In [2]:
from pathlib import Path

artifacts = Path('artifacts')
print((artifacts / 'benchmark_report.md').read_text(encoding='utf-8'))

# Wait-Time Benchmark Report

## Summary

- Acceptance passed: `True`
- Selected trial: `large`
- Best epoch: `8`
- Suppress threshold: `0.42`

## Training Coverage

- Train size: `72000`
- Scenario mix: `{'urgent_incident': 7028, 'accountability_checkin': 7038, 'support_followup': 7746, 'polite_close': 6204, 'scheduled_reminder': 9977, 'async_deliverable': 8283, 'sensitive_space': 5479, 'do_not_disturb': 7019, 'clarification_needed': 6983, 'order_status': 6243}`

## Model Metrics

| Split | Practical | Adaptive-60s | Adaptive-300s | Short MAE(s) | P90 Error(s) | Follow-up F1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| test_iid | 0.969 | 0.962 | 0.984 | 1.5 | 520.0 | 0.996 |
| test_lexical | 0.975 | 0.965 | 0.976 | 0.0 | 630.2 | 1.000 |
| test_stress | 0.967 | 0.971 | 0.984 | 3.8 | 300.8 | 0.999 |
| overall | 0.970 | 0.966 | 0.982 | 1.8 | 483.7 | 0.998 |

## Keyword Baseline

| Split | Practical | Adaptive-60s | Adaptive-300s | Short MAE(s) | P90 Error(s) | Follow-up F1 |
| -

In [3]:
import torch
from wait_model import ConversationExample, humanize_wait, load_model_bundle, predict_wait_time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, featurizer, meta = load_model_bundle('artifacts/wait_policy_bundle.pt', device=device)

example = ConversationExample(
    turns=[
        {'role': 'user', 'text': '我會忙到明天下午，如果我還沒把方案發出去，你就提醒我。'},
        {'role': 'assistant', 'text': '好，我先不打擾，等到明天下午如果還沒收到，我再主動跟進。'},
    ],
    hour_local=20,
    weekday=2,
)

prediction = predict_wait_time(model, featurizer, example, device=device, suppress_threshold=meta['suppress_threshold'])
print(prediction)
print('predicted_wait =', 'suppress' if prediction['suppress'] else humanize_wait(prediction['wait_seconds']))

{'bucket_id': 24, 'wait_seconds': 68400.0, 'suppress': False, 'label': '18h-24h', 'probabilities': [8.21555750007974e-06, 1.089820852939738e-05, 3.948856465285644e-05, 5.475975194713101e-05, 4.014047590317205e-06, 0.00013191744801588356, 6.112629762355937e-06, 1.813082599255722e-05, 0.0005864494014531374, 4.706529216491617e-05, 0.00012302628601901233, 8.972067735157907e-05, 0.00013559374201577157, 1.8121783796232194e-05, 0.0012965502683073282, 0.00028653722256422043, 0.00012193984730402008, 0.00046696935896761715, 0.0002951931091956794, 0.0013298640260472894, 5.391664308262989e-05, 0.00012867958866991103, 0.0001531399175291881, 0.02255146950483322, 0.956940770149231, 0.014527766965329647, 0.00046671958989463747, 8.662746949994471e-06, 1.2945943126396742e-05, 7.044349331408739e-06, 5.437500021798769e-06, 7.283472223207355e-05]}
predicted_wait = 19.0h


In [4]:
from pathlib import Path

bundle_path = Path('artifacts/wait_policy_bundle.pt')
state_path = Path('artifacts/wait_policy_state.pt')
metrics_path = Path('artifacts/metrics.json')

print('bundle:', bundle_path.resolve())
print('state :', state_path.resolve())
print('metrics:', metrics_path.resolve())

try:
    from google.colab import files
    files.download(str(bundle_path))
    files.download(str(state_path))
    files.download(str(metrics_path))
except Exception:
    print('Not running inside Colab download context. Files are already saved locally under artifacts/.')

bundle: C:\Users\33835\ashley\artifacts\wait_policy_bundle.pt
state : C:\Users\33835\ashley\artifacts\wait_policy_state.pt
metrics: C:\Users\33835\ashley\artifacts\metrics.json
Not running inside Colab download context. Files are already saved locally under artifacts/.
